# RetailMind AI — Day 5: Neural Networks and ANN (Keras)

This increment introduces a small feed-forward neural network for customer churn and compares it head-to-head with the Day 4 classical champion. The ANN is an experiment: it is not assumed to be better.

Run from the repository root with the same Python environment used by the project. Install `requirements.txt` first. The Telco CSV should be in `data/`, and the Day 4 champion should exist in `models/`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import tensorflow as tf

from src.data_preprocessing import get_train_test_data
from src.day5_ann import train_and_benchmark_ann, predict_day5

print("TensorFlow:", tf.__version__)
print("Project:", PROJECT_ROOT)

## 1. Warm-up: Keras on MNIST

MNIST is a small image-classification problem. This demo shows a flattened input, a hidden dense layer with ReLU, a ten-class softmax output, and a standard Keras training loop. It uses a subset to keep the warm-up short. The dataset downloads the first time it is requested.

In [ ]:
(mnist_x_train, mnist_y_train), (mnist_x_test, mnist_y_test) = (
    tf.keras.datasets.mnist.load_data()
)

mnist_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
mnist_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
mnist_history = mnist_model.fit(
    mnist_x_train[:10000],
    mnist_y_train[:10000],
    validation_split=0.1,
    epochs=2,
    batch_size=64,
    verbose=2,
)
mnist_loss, mnist_accuracy = mnist_model.evaluate(
    mnist_x_test, mnist_y_test, verbose=0
)
print(f"MNIST test accuracy: {mnist_accuracy:.4f}")

## 2. Build the churn ANN

The reusable model uses Keras preprocessing layers: numeric values are normalized using training rows, and each categorical column is represented with a `StringLookup` vocabulary learned from training rows. Two ReLU hidden layers feed a sigmoid churn probability. `EarlyStopping` monitors validation loss and restores the best weights.

The fixed test set is used only for the final benchmark, not for early stopping or model selection.

In [ ]:
X_train, X_test, y_train, y_test = get_train_test_data()
ann_model, comparison, ann_history = train_and_benchmark_ann(
    X_train,
    y_train,
    X_test,
    y_test,
    epochs=100,
    batch_size=32,
    patience=8,
)
display(comparison)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(ann_history.history["loss"], label="Training loss")
plt.plot(ann_history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.title("Day 5 — ANN learning curve")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
print("Best validation loss:", min(ann_history.history["val_loss"]))

## 3. Interpret and use the benchmark

Compare recall, precision, F1, and ROC-AUC at the shared 0.5 threshold. A neural network has not demonstrated added value merely because it has more layers: use validation evidence and the same held-out split. For modest tabular data, a classical model may remain preferable for similar predictive quality with less training and serving complexity.

The saved Keras model is `models/churn_day5_ann.keras`; its numeric imputation and feature-column metadata are saved alongside it. The comparison is written to `reports/day5_ann_vs_classical.csv`.

In [ ]:
display(predict_day5(X_test.head()))

## Run from the terminal

Run only Day 5 after completing Day 4:

```shell
python -m src.day5_ann
```

Run the complete RetailMind pipeline through Day 5:

```shell
python -m src.run_pipeline
```